## Create Streaming Sources

Streaming Table is a Unity Catalog managed table with built-in support for incremental data processing. They ingest incrementally via Auto Loader, and only new files are processed on each refresh 

Supports data expectations, which is similar but not exactly dbt tests:
- dbt tests = "check the table after it's built."
- Expectations = "guard the door as rows come in."

## Courses

In [0]:
CREATE OR REFRESH STREAMING TABLE language_app.bronze.raw_courses (
  -- warn: reference data, tell me if it ever goes wrong
  CONSTRAINT valid_course_id
    EXPECT (course_id IS NOT NULL),

  CONSTRAINT valid_level
    EXPECT (level IN ('A1', 'A2', 'B1', 'B2', 'C1', 'C2')),

  CONSTRAINT valid_total_lessons
    EXPECT (total_lessons > 0)
)
  TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.enableRowTracking' = 'true',
    'delta.enableDeletionVectors' = 'true'
  ) AS
(
  SELECT
    course_id,
    language,
    level,
    course_name,
    total_lessons,
    _rescued_data
  FROM STREAM
    read_files("/Volumes/language_app/source/raw_files/courses/", format => "csv", header => true)
);

## Lesson Attempts

In [0]:
CREATE OR REFRESH STREAMING TABLE language_app.bronze.raw_lesson_attempts (
  -- fail: a missing ID means the feed is broken, so stop the update
  CONSTRAINT valid_attempt_id
    EXPECT (attempt_id IS NOT NULL) ON VIOLATION FAIL UPDATE,

  -- drop: can't attribute the attempt to anyone
  CONSTRAINT valid_learner_id
    EXPECT (learner_id IS NOT NULL) ON VIOLATION DROP ROW,

  -- drop: out-of-range scores would distort gold averages
  CONSTRAINT valid_score
    EXPECT (score BETWEEN 0 AND 100) ON VIOLATION DROP ROW,

  -- drop: future dates mean a clock error
  CONSTRAINT valid_attempted_at
    EXPECT (attempted_at <= current_timestamp()) ON VIOLATION DROP ROW,

  -- ===== REMEMBER: Warn is the default =====
  -- warn: keep the row, fix in silver
  CONSTRAINT valid_minutes
    EXPECT (minutes_spent >= 0),

  -- warn: early warning for new device values
  CONSTRAINT valid_device
    EXPECT (device IN ('ios', 'android', 'web'))
)
  TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.enableRowTracking' = 'true',
    'delta.enableDeletionVectors' = 'true'
  )
AS
(
  SELECT
    attempt_id,
    learner_id,
    course_id,
    lesson_number,
    attempted_at,
    score,
    minutes_spent,
    completed,
    device,
    _rescued_data
  FROM STREAM
    read_files(
      "/Volumes/language_app/source/raw_files/lesson_attempts/",
      format => "csv",
      header => true
    )
);